In [1]:
import datetime as dt
import time

import pandas as pd

from utilities.preprocessor import *
from utilities.values import *

In [2]:
data_dir = './data/'
dataset_preprocessor = DatasetPreprocessor()

#### 1. Merge the separate raw datafiles

In [3]:
# combine all the raw files
raw_data_dir_paths = [data_dir + 'raw/2020', data_dir + 'raw/2021',]
full_dataset_df = dataset_preprocessor.read_all_files_in_dir(raw_data_dir_paths)

# sort by increasing start and end dates
full_dataset_df = full_dataset_df.sort_values(['starttime', 'stoptime'])
print('Total number of rows:', len(full_dataset_df))

Total number of rows: 393312


### 2. EDA

In [4]:
# check for missing values - None
def print_missing_values(df):
    num_rows_with_missing_values = df.isnull().any(axis=1).sum()
    print(df.isna().sum())
    print('Total rows:', len(df))
    print('Rows with missing values:', num_rows_with_missing_values)
    print('Proportion: {} %'.format(round(num_rows_with_missing_values/len(df) *100, 4)))

print_missing_values(full_dataset_df)

tripduration               0
starttime                  0
stoptime                   0
start station id           0
start station name         0
start station latitude     0
start station longitude    0
end station id             0
end station name           0
end station latitude       0
end station longitude      0
bikeid                     0
usertype                   0
birth year                 0
gender                     0
dtype: int64
Total rows: 393312
Rows with missing values: 0
Proportion: 0.0 %


In [5]:
# look at distributions

### 2. Preprocessing the dataset

#### 2a. Extract date features

In [6]:
# force starttime into datetime format
full_dataset_df['starttime'] = pd.to_datetime(full_dataset_df['starttime'])

# create new date features
full_dataset_df[DATE] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_date)
full_dataset_df[MONTH_OF_YEAR] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_month_of_year)
full_dataset_df[WEEK_OF_YEAR] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_week_of_year)
full_dataset_df[DAY_OF_WEEK] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_day_of_week)
full_dataset_df[HOUR_OF_DAY] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_hour_of_day)
full_dataset_df[IS_PUBLIC_HOLIDAY] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_is_public_holiday)

In [7]:
# drop columns that will not be used
columns_to_drop = [
    'tripduration',
    'starttime',
    'stoptime',
    'start station id',
    'end station id',
    'end station name',
    'end station latitude',
    'end station longitude',
    'bikeid',
    'usertype',
    'birth year',
    'gender'
]
full_dataset_df = full_dataset_df.drop(columns=columns_to_drop)

In [8]:
full_rental_counts_df = (
    full_dataset_df
    .groupby(list(full_dataset_df.columns), dropna = False)
    .size()
    .reset_index(name = NUM_RENTALS)
)

# save to file
full_rental_counts_df.to_csv(data_dir + 'intermediate_processing/full_rental_counts.csv', index=False)

In [9]:
full_rental_counts_df

,start station name,start station latitude,start station longitude,date,month_of_year,week_of_year,day_of_week,hour_of_day,is_public_holiday,number_of_rentals
0,5 Corners Library,40.734961,-74.059503,2020-01-01,1,1,2,16,1,1
1,5 Corners Library,40.734961,-74.059503,2020-01-01,1,1,2,8,1,1
2,5 Corners Library,40.734961,-74.059503,2020-01-02,1,1,3,12,0,1
3,5 Corners Library,40.734961,-74.059503,2020-01-02,1,1,3,13,0,1
4,5 Corners Library,40.734961,-74.059503,2020-01-02,1,1,3,18,0,3
...,...,...,...,...,...,...,...,...,...,...
176354,York St,40.716490,-74.041050,2021-03-31,3,13,2,20,0,1
176355,York St,40.716490,-74.041050,2021-04-01,4,13,3,19,0,1
176356,York St,40.716490,-74.041050,2021-04-01,4,13,3,20,0,1
176357,York St,40.716490,-74.041050,2021-04-01,4,13,3,23,0,1


#### 2b. Extract weather features

In [10]:
# get the location and dates to query
weather_search_df = full_rental_counts_df[[STATION, LATITUDE, LONGITUDE, DATE]]
weather_search_df = weather_search_df.drop_duplicates()
stations = set(weather_search_df[STATION])

In [11]:
# call the weather API for each location
weather_df = pd.DataFrame()

for i, station in enumerate(stations, start = 1):
    location_df = weather_search_df[weather_search_df[STATION] == station]
    location_df = location_df.sort_values(DATE)

    params = {
        START_DATE: location_df[DATE].iloc[0],
        END_DATE: location_df[DATE].iloc[-1],
        STATION: station,
        LATITUDE: location_df[LATITUDE].iloc[0],
        LONGITUDE: location_df[LONGITUDE].iloc[0]
    }

    # get historical weather data
    station_weather_history_df = dataset_preprocessor.get_hourly_weather(params)
    weather_df = pd.concat([weather_df, station_weather_history_df])

    # pause for 1 min after every 20 calls to stay under the API rate limit
    if i % 20 == 0:
        print(f"{i} stations done, sleeping for 60s...")
        time.sleep(60)

# save results
weather_df.to_csv(data_dir + 'intermediate_processing/historical_weather_features.csv', index=False)

20 stations done, sleeping for 60s...
40 stations done, sleeping for 60s...


#### 2c. Merge the citibike and weather features

#### 2d. Train - Validation - Test split